# Our first look at the credit-card data

**WHAT:** See the first five actual customer records.

This notebook is in the project folder so it can read the existing file in `data/raw/`. Nothing is downloaded or changed in the dataset. Run cells from top to bottom with **Shift+Enter**. Select the existing **Python 3.10** environment as the kernel (the Python program that runs notebook code).

## 1. Make table tools available

**WHICH code:** `import pandas as pd`

**pandas** is a Python library for working with tables. `pd` is the short name we give it.

**OUTPUT:** No visible output is expected when this succeeds.

In [1]:
import pandas as pd

## 2. Read our saved table

**WHICH code:** `pd.read_csv(...)` reads the existing CSV. The quoted text is the file location relative to the project folder.

A **DataFrame** is a table in Python. A **variable** is a name referring to something in our program; `df` refers to the loaded table. `=` assigns the table to that name.

**OUTPUT:** No visible output is expected. The table is now available as `df`.

In [2]:
df = pd.read_csv("data/raw/default_credit_card_clients.csv")

## 3. Look at five records

**WHICH code:** `df.head()` returns the first five rows. A notebook displays the last expression automatically, so no `print()` is needed.

**WHAT the output means:** Each row is one customer record. Each column is one field. For example, `X1` is the credit limit in New Taiwan dollars (NT$). The leftmost labels 0 through 4 are pandas row labels, separate from `ID`. An ellipsis (`...`) means columns are hidden to fit the display, not removed. `head()` does not delete the remaining records.

In [3]:
df.head()

,ID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## Understand the table's size

**WHAT:** Find the number of customer records and fields in the dataset.

**WHY:** Before interpreting a dataset, we need to know its size. A row represents one customer record and a column represents one field.

**CODE:** `df.shape` returns a pair: `(number of rows, number of columns)`.

In [4]:
df.shape

(30000, 25)

**WHAT THE ACTUAL OUTPUT MEANS:** `(30000, 25)` means 30,000 customer records and 25 columns. The 25 columns include `ID`, 23 columns of recorded information, and `Y`, the recorded future default outcome. This output does not describe 30,000 customers today; it describes the historical dataset.

## See every column name

**WHAT:** See the exact names used in the CSV.

**WHY:** The preview used `...` because not all columns fit on screen. We need the exact names before interpreting them from the official documentation.

**CODE:** `df.columns` refers to the column labels. `.tolist()` converts those labels into a regular Python list so they display without truncation.

In [5]:
df.columns.tolist()

['ID', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'Y']

**WHAT THE ACTUAL OUTPUT MEANS:** The CSV has one identifier column (`ID`), fields named `X1` to `X23`, and one column named `Y`. `X1` to `X23` are placeholders in this file, so their business meanings must come from the verified UCI documentation.

## The prediction target: `Y`

**WHAT IT IS:** UCI defines `Y` as `default payment next month`. It is the recorded outcome: `1` means default payment Yes and `0` means default payment No.

**WHY IT MATTERS FOR CREDIT RISK:** This is the event the project aims to estimate before it happens. The documented history runs to September 2005, so `next month` is naturally read as October 2005; this calendar month is an inference from the documented dates, while UCI itself labels the outcome only as “next month.”

`Y` is called the **target** because it is the answer the model would try to predict. The other recorded columns are possible **input features** because they describe information available before the outcome. A future model must not use `Y` as an input: it would give the answer away.

## Verified UCI column map and groups

The following meanings come from the stored official UCI metadata.

| Group | CSV columns | UCI meaning |
| --- | --- | --- |
| Identifier | `ID` | Record identifier |
| Customer information | `X1`–`X5` | Credit limit, sex, education, marital status, age |
| Repayment status | `X6`–`X11` | September to April 2005 repayment status |
| Bill amounts | `X12`–`X17` | September to April 2005 bill statements, in NT$ |
| Payment amounts | `X18`–`X23` | September to April 2005 previous-payment amounts, in NT$ |
| Future outcome | `Y` | Default payment next month: Yes = 1, No = 0 |

For the detailed fields below, the official names are: `X1` = `LIMIT_BAL`, `X6` = `PAY_0`, `X12` = `BILL_AMT1`, and `X18` = `PAY_AMT1`.

## Five fields for this lesson

- **`ID` — customer record identifier.** It distinguishes records. It has no currency or month. A larger ID only means a larger identifier; it is not a measure of credit risk.
- **`X1` / `LIMIT_BAL` — amount of credit granted.** UCI gives its unit as NT$ and says it includes individual consumer credit and supplementary family credit. UCI does not assign it a particular month. A larger or smaller number means a larger or smaller recorded granted-credit amount; it does not by itself mean higher or lower default risk. UCI does not document a special interpretation for zero or negative values.
- **`X6` / `PAY_0` — repayment status in September 2005.** This is the most recent repayment-status field. UCI documents `-1` as paid duly; `1` through `8` as one through eight months of payment delay; and `9` as nine months or more of delay. Therefore higher documented positive values mean longer recorded delay. UCI does not define `0` or `-2`, so we must not guess their meanings.
- **`X12` / `BILL_AMT1` — September 2005 bill statement amount.** Its unit is NT$. A higher or lower value is only a numerically higher or lower recorded bill statement. UCI does not document what zero or a negative bill amount means, nor does it define a risk interpretation for these amounts.
- **`X18` / `PAY_AMT1` — amount paid in September 2005.** Its unit is NT$. A higher or lower value is only a numerically higher or lower recorded payment amount. UCI does not document a special meaning for zero or negative values.

**Important timing point:** UCI labels both `X12` and `X18` as September 2005. It does not document their exact within-month billing and payment timing or pairing. Therefore we should not call `X18 / X12` a payment ratio yet.

## View only these fields

**WHAT:** View the identifier, credit limit, latest repayment status, latest bill amount, latest payment amount, and outcome together.

**WHY:** This connects the coded field names to actual recorded values without changing any data.

**CODE:** The outer double square brackets select a list of named columns; `.head()` limits the display to the first five records.

In [6]:
df[["ID", "X1", "X6", "X12", "X18", "Y"]].head()

   ID      X1  X6    X12   X18  Y
0   1   20000   2   3913     0  1
1   2  120000  -1   2682     0  1
2   3   90000   0  29239  1518  0
3   4   50000   0  46990  2000  0
4   5   50000  -1   8617  2000  0

**WHAT THE ACTUAL OUTPUT MEANS:** For record ID 1, `X1=20000` means NT$20,000 granted credit; `X6=2` means a documented two-month payment delay in September 2005; `X12=3913` is a September bill statement of NT$3,913; `X18=0` is a recorded September payment amount of NT$0; and `Y=1` is a recorded default payment next month.

For ID 2, `X6=-1` is documented as paid duly. For IDs 3 and 4, `X6=0` is present, but UCI does not define that code, so no interpretation is assigned here. These are historical observed records, not predictions.

**Question:** Why must we not use `Y` as an input feature when predicting `Y`?

# Check whether the data is ready to analyse

Before making conclusions or training a model, inspect the data structure. A **missing value** is an absent value. A **duplicate customer record** would mean the same identifier appears more than once. A **data type** tells Python what sort of value a column stores, such as an integer. An **invalid or unexpected value** is a value that conflicts with verified documentation or a clearly defined business rule; it is not simply a zero or negative number.

These checks help reveal problems that could distort analysis or model results. They do not prove that every value is correct, and no data will be changed in this lesson. Negative repayment-status values must be interpreted only from official documentation: UCI documents `-1` but does not define every negative code.

## Data types and non-missing counts

**WHAT QUESTION THIS ANSWERS:** How many non-missing values does each column have, and what data type did pandas assign?

**WHY IT MATTERS:** A column with fewer recorded values may need investigation. Data types affect which calculations and checks are meaningful.

**CODE:** `df.info()` prints a compact summary of the DataFrame.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   ID      30000 non-null  int64
 1   X1      30000 non-null  int64
 2   X2      30000 non-null  int64
 3   X3      30000 non-null  int64
 4   X4      30000 non-null  int64
 5   X5      30000 non-null  int64
 6   X6      30000 non-null  int64
 7   X7      30000 non-null  int64
 8   X8      30000 non-null  int64
 9   X9      30000 non-null  int64
 10  X10     30000 non-null  int64
 11  X11     30000 non-null  int64
 12  X12     30000 non-null  int64
 13  X13     30000 non-null  int64
 14  X14     30000 non-null  int64
 15  X15     30000 non-null  int64
 16  X16     30000 non-null  int64
 17  X17     30000 non-null  int64
 18  X18     30000 non-null  int64
 19  X19     30000 non-null  int64
 20  X20     30000 non-null  int64
 21  X21     30000 non-null  int64
 22  X22     30000 non-null  int64
 23  X23     300

**WHAT THE ACTUAL OUTPUT MEANS:** Every listed column has 30,000 non-null values, matching the 30,000 rows. Pandas read all 25 columns as `int64`, an integer data type. This is a structural result only; an integer can still be unexpected and needs documentation before interpretation.

## Missing-value counts

**WHAT QUESTION THIS ANSWERS:** How many missing values are in each column?

**WHY IT MATTERS:** Missing values can affect summaries and models. We check them before deciding whether any treatment is needed.

**CODE:** `df.isna()` marks missing values as `True`; `.sum()` counts those `True` values in each column.

In [8]:
df.isna().sum()

ID     0
X1     0
X2     0
X3     0
X4     0
X5     0
X6     0
X7     0
X8     0
X9     0
X10    0
X11    0
X12    0
X13    0
X14    0
X15    0
X16    0
X17    0
X18    0
X19    0
X20    0
X21    0
X22    0
X23    0
Y      0
dtype: int64

**WHAT THE ACTUAL OUTPUT MEANS:** Every column has a count of 0, so pandas found no missing values in this CSV. No values are filled, removed, or transformed.

## Duplicate identifiers

**WHAT QUESTION THIS ANSWERS:** Does any `ID` appear after an earlier occurrence of the same ID?

**WHY IT MATTERS:** `ID` is the record identifier. Repeated IDs could mean duplicated customer records and could give those records extra influence in analysis.

**CODE:** `.duplicated()` marks repeated values after their first occurrence; `.sum()` counts them.

In [9]:
df["ID"].duplicated().sum()

0

**WHAT THE ACTUAL OUTPUT MEANS:** The result is 0. There are no repeated IDs in this file. This checks identifier duplication only; it does not establish that all customer details are unique or correct.

## Count the recorded outcomes

**WHAT QUESTION THIS ANSWERS:** How many records have each value of the target `Y`?

**WHY IT MATTERS:** It shows whether the future default outcome is balanced or imbalanced before any model is evaluated.

**CODE:** `value_counts()` counts each distinct value in `Y`.

In [10]:
df["Y"].value_counts()

Y
0    23364
1     6636
Name: count, dtype: int64

**WHAT THE ACTUAL OUTPUT MEANS:** 23,364 records have `Y=0` (no default payment next month) and 6,636 have `Y=1` (default payment next month). The two outcomes are not equally common.

## Convert outcome counts to proportions

**WHAT QUESTION THIS ANSWERS:** What share of records has each outcome?

**WHY IT MATTERS:** Proportions make the imbalance easier to judge than raw counts alone. If most customers have `Y=0`, accuracy alone can be misleading: a model could predict no default for everyone and receive many answers right while finding no defaults.

**CODE:** `normalize=True` asks `value_counts()` to divide each count by the total number of records.

In [11]:
df["Y"].value_counts(normalize=True)

Y
0    0.7788
1    0.2212
Name: proportion, dtype: float64

**WHAT THE ACTUAL OUTPUT MEANS:** `Y=0` accounts for 0.7788, or 77.88%, of records. `Y=1` accounts for 0.2212, or 22.12%. This is an imbalanced target because the two outcomes are not equally frequent.

**Question:** If 80% of customers do not default, why could a model that predicts ‘no default’ for every customer look accurate but still be useless?

# Exploratory Data Analysis (EDA)

This section describes the historical UCI Taiwan credit-card records before any new modelling work. It reuses the project's `load_data()` function, does not use demographic columns `X2`–`X5`, and does not train or change any model. Charts are saved to `reports/figures/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from data_loading import load_data

df = load_data()
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

COLORS = {'No default': '#4C78A8', 'Default next month': '#E45756'}
sns.set_theme(style='whitegrid', context='notebook')

def save_figure(filename):
    plt.tight_layout()
    plt.savefig(FIGURES / filename, dpi=200, bbox_inches='tight')
    plt.show()

## EDA 1 — Target-class balance

1. **What we are checking:** How many records have a recorded default-payment outcome next month (`Y=1`) versus no default (`Y=0`), in counts and percentages.

2. **Why it matters in a credit-risk problem:** Defaults are less common than non-defaults, so a model can look accurate simply by predicting no default for everyone.

3. **The code:** The cell groups `Y`, calculates its count and percentage, and draws a labelled bar chart.

4. **How to read the output:** Read both labels: the count shows the data volume and the percentage shows the class balance.

5. **One honest business insight or limitation:** This is a historical outcome rate, not a forecast of a current bank's default rate.

In [ ]:
target_summary = (df['Y'].value_counts().sort_index().rename_axis('Y').reset_index(name='count'))
target_summary['percentage'] = 100 * target_summary['count'] / len(df)
target_summary['outcome'] = target_summary['Y'].map({0: 'No default', 1: 'Default next month'})
display(target_summary[['outcome', 'count', 'percentage']].style.format({'percentage': '{:.2f}%'}))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(target_summary['outcome'], target_summary['count'], color=[COLORS['No default'], COLORS['Default next month']])
ax.set(title='Historical next-month default outcome balance', xlabel='', ylabel='Number of records')
for bar, row in zip(bars, target_summary.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{row.count:,}\n({row.percentage:.2f}%)', ha='center', va='bottom')
save_figure('01_target_class_balance.png')

## EDA 2 — Credit-limit distribution

1. **What we are checking:** The overall distribution of granted credit (`X1`) and how it differs between the two recorded outcomes.

2. **Why it matters in a credit-risk problem:** Credit limit is an account characteristic available before the future outcome and may help describe portfolio composition.

3. **The code:** The cell creates an overall histogram and an outcome-split histogram using the same NT$ scale.

4. **How to read the output:** A right-skewed distribution has many lower values and fewer very large values. Compare the shapes, not only the tallest bar.

5. **One honest business insight or limitation:** Different distributions show association in this historical sample; they do not show that changing a credit limit would cause default risk to change.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=df, x='X1', bins=40, color='#4C78A8', ax=axes[0])
axes[0].set(title='Granted credit distribution: all records', xlabel='Granted credit (NT$)', ylabel='Number of records')
plot_data = df.assign(outcome=df['Y'].map({0: 'No default', 1: 'Default next month'}))
sns.histplot(data=plot_data, x='X1', hue='outcome', bins=40, stat='density', common_norm=False, element='step', fill=True, palette=COLORS, ax=axes[1])
axes[1].set(title='Granted credit distribution by recorded outcome', xlabel='Granted credit (NT$)', ylabel='Density')
save_figure('02_credit_limit_distribution.png')

df.groupby('Y')['X1'].agg(['count', 'mean', 'median', 'min', 'max']).rename(index={0: 'No default', 1: 'Default next month'}).round(2)

## EDA 3 — Default rate across credit-limit bands

1. **What we are checking:** The historical default-payment rate within broad, readable granted-credit bands.

2. **Why it matters in a credit-risk problem:** Segment-level rates help a risk team understand where observed outcomes were more or less common, while retaining the group sizes needed for context.

3. **The code:** The code creates four non-overlapping bands, then calculates count and mean of binary `Y`; the mean of `Y` is the observed default rate.

4. **How to read the output:** Compare both the bar height and the labelled record count. A rate from a small group is less stable than a rate from a large group.

5. **One honest business insight or limitation:** The bands are descriptive choices for EDA, not approved lending rules or causal thresholds.

In [ ]:
limit_bins = [-np.inf, 50_000, 100_000, 200_000, np.inf]
limit_labels = ['Below NT$50k', 'NT$50k–99,999', 'NT$100k–199,999', 'NT$200k or more']
df_eda = df.assign(credit_limit_band=pd.cut(df['X1'], bins=limit_bins, labels=limit_labels, right=False))
limit_rates = df_eda.groupby('credit_limit_band', observed=False)['Y'].agg(records='size', observed_default_rate='mean').reset_index()
display(limit_rates.style.format({'observed_default_rate': '{:.2%}'}))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(limit_rates['credit_limit_band'].astype(str), limit_rates['observed_default_rate'], color='#E45756')
ax.set(title='Historical default-payment rate by granted-credit band', xlabel='Granted-credit band', ylabel='Observed default-payment rate', ylim=(0, limit_rates['observed_default_rate'].max() * 1.25))
ax.yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
ax.tick_params(axis='x', rotation=15)
for bar, row in zip(bars, limit_rates.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{row.observed_default_rate:.1%}\nn={row.records:,}', ha='center', va='bottom')
save_figure('03_default_rate_credit_limit_band.png')

## EDA 4 — Default rate by repayment-status history (`X6`–`X11`)

1. **What we are checking:** The historical default rate for each observed code in every repayment-status column, from the latest month (`X6`) back to the earliest (`X11`).

2. **Why it matters in a credit-risk problem:** Repayment history is central to credit risk because it describes recorded prior payment status before the next-month outcome.

3. **The code:** The code preserves every observed code, labels only documented values (`-1`, `1`–`9`), and labels `-2` and `0` as ‘Undocumented code’ rather than guessing their meaning.

4. **How to read the output:** Each point is a group’s observed rate. Read its `n=` label alongside its height, especially for rare higher-delay codes.

5. **One honest business insight or limitation:** Very small groups can have volatile rates, and the supplied UCI documentation does not define codes `-2` and `0`.

In [ ]:
status_columns = [f'X{i}' for i in range(6, 12)]
status_months = {'X6': 'Sep 2005', 'X7': 'Aug 2005', 'X8': 'Jul 2005', 'X9': 'Jun 2005', 'X10': 'May 2005', 'X11': 'Apr 2005'}
def status_label(value):
    if value == -1:
        return '-1: paid duly (documented)'
    if 1 <= value <= 8:
        return f'{value}: {value} month(s) delay (documented)'
    if value == 9:
        return '9: 9+ months delay (documented)'
    return f'{value}: undocumented code'

status_rates = pd.concat([
    df.groupby(column)['Y'].agg(records='size', observed_default_rate='mean').reset_index().rename(columns={column: 'status_code'}).assign(variable=column, month=status_months[column])
    for column in status_columns
], ignore_index=True)
status_rates['status_label'] = status_rates['status_code'].map(status_label)
display(status_rates.sort_values(['variable', 'status_code']).style.format({'observed_default_rate': '{:.2%}'}))

fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharey=True)
for ax, column in zip(axes.flat, status_columns):
    subset = status_rates[status_rates['variable'] == column].sort_values('status_code')
    ax.plot(subset['status_code'], subset['observed_default_rate'], marker='o', color='#6F4E7C')
    for row in subset.itertuples():
        ax.annotate(f'n={row.records:,}', (row.status_code, row.observed_default_rate), xytext=(0, 5), textcoords='offset points', ha='center', fontsize=7)
    ax.set(title=f'{column}: {status_months[column]}', xlabel='Observed repayment-status code', ylabel='Observed default rate')
    ax.yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
fig.suptitle('Historical default-payment rate by repayment-status code', y=1.02, fontsize=15)
save_figure('04_default_rate_by_repayment_status.png')

## EDA 5 — Latest repayment status (`X6`) and next-month outcome

1. **What we are checking:** The latest available repayment-status code, its record count, and its observed next-month default-payment rate.

2. **Why it matters in a credit-risk problem:** The most recent status is often operationally useful for prioritising account review, provided its meaning is documented and the policy is validated.

3. **The code:** The code groups the September 2005 status `X6`, calculates counts and rates, and labels each code honestly.

4. **How to read the output:** The left chart shows how common a code is; the right chart shows the corresponding outcome rate. Use both together.

5. **One honest business insight or limitation:** This is the strongest descriptive relationship in this EDA, but it still does not establish a causal effect or an action rule.

In [ ]:
latest_status = status_rates[status_rates['variable'] == 'X6'].sort_values('status_code').copy()
display(latest_status[['status_code', 'status_label', 'records', 'observed_default_rate']].style.format({'observed_default_rate': '{:.2%}'}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(latest_status['status_code'].astype(str), latest_status['records'], color='#4C78A8')
axes[0].set(title='Latest repayment-status code frequency', xlabel='X6 observed code', ylabel='Number of records')
bars = axes[1].bar(latest_status['status_code'].astype(str), latest_status['observed_default_rate'], color='#E45756')
axes[1].set(title='Next-month observed default rate by latest status', xlabel='X6 observed code', ylabel='Observed default-payment rate')
axes[1].yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
for bar, row in zip(bars, latest_status.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{row.observed_default_rate:.1%}', ha='center', va='bottom', fontsize=8)
save_figure('05_latest_repayment_status_outcome.png')

## EDA 6 — Bill amounts (`X12`–`X17`)

1. **What we are checking:** Summary statistics and distributions for six monthly bill-statement amounts.

2. **Why it matters in a credit-risk problem:** Bill amounts describe the recorded account statements before the outcome and can reveal scale, skewness, and unusual values requiring careful handling.

3. **The code:** The code displays descriptive statistics, then uses a log-like `symlog` axis so zero and negative values remain visible without dropping records.

4. **How to read the output:** The table gives centre and spread; the boxplots show median, spread, and extreme values. The symlog scale means spacing is not a simple linear NT$ scale.

5. **One honest business insight or limitation:** A bill amount is not automatically current debt, purchases, or a validated measure of utilisation.

In [ ]:
bill_columns = [f'X{i}' for i in range(12, 18)]
bill_months = {'X12': 'Sep', 'X13': 'Aug', 'X14': 'Jul', 'X15': 'Jun', 'X16': 'May', 'X17': 'Apr'}
bill_summary = df[bill_columns].describe(percentiles=[.25, .5, .75]).T.rename(index=bill_months)
display(bill_summary.style.format('{:,.0f}'))

bill_long = df[bill_columns].rename(columns=bill_months).melt(var_name='month', value_name='bill_amount_ntd')
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=bill_long, x='month', y='bill_amount_ntd', color='#72B7B2', showfliers=False, ax=axes[0])
axes[0].set_yscale('symlog', linthresh=1_000)
axes[0].set(title='Bill statement amounts by month', xlabel='Statement month in 2005', ylabel='Bill amount (NT$, symlog scale)')
sns.histplot(data=bill_long, x='bill_amount_ntd', bins=60, color='#72B7B2', ax=axes[1])
axes[1].set_xscale('symlog', linthresh=1_000)
axes[1].set(title='All recorded bill statement amounts', xlabel='Bill amount (NT$, symlog scale)', ylabel='Number of values')
save_figure('06_bill_amount_summary_distributions.png')

## EDA 7 — Payment amounts (`X18`–`X23`)

1. **What we are checking:** Summary statistics and distributions for six monthly recorded payment amounts.

2. **Why it matters in a credit-risk problem:** Payment amounts may contain useful historical behaviour, but their interpretation must respect the source’s missing within-month timing details.

3. **The code:** The code displays descriptive statistics and plots all six payment columns with zero preserved on a symlog scale.

4. **How to read the output:** A concentration near zero means many recorded amounts are small or zero; compare monthly medians and distributions rather than treating one amount as a complete repayment record.

5. **One honest business insight or limitation:** A recorded payment is not necessarily a minimum payment, full repayment, or proof of timely payment.

In [ ]:
payment_columns = [f'X{i}' for i in range(18, 24)]
payment_months = {'X18': 'Sep', 'X19': 'Aug', 'X20': 'Jul', 'X21': 'Jun', 'X22': 'May', 'X23': 'Apr'}
payment_summary = df[payment_columns].describe(percentiles=[.25, .5, .75]).T.rename(index=payment_months)
display(payment_summary.style.format('{:,.0f}'))

payment_long = df[payment_columns].rename(columns=payment_months).melt(var_name='month', value_name='payment_amount_ntd')
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=payment_long, x='month', y='payment_amount_ntd', color='#F2CF5B', showfliers=False, ax=axes[0])
axes[0].set_yscale('symlog', linthresh=1_000)
axes[0].set(title='Recorded payment amounts by month', xlabel='Payment month in 2005', ylabel='Payment amount (NT$, symlog scale)')
sns.histplot(data=payment_long, x='payment_amount_ntd', bins=60, color='#F2CF5B', ax=axes[1])
axes[1].set_xscale('symlog', linthresh=1_000)
axes[1].set(title='All recorded payment amounts', xlabel='Payment amount (NT$, symlog scale)', ylabel='Number of values')
save_figure('07_payment_amount_summary_distributions.png')

## EDA 8 — Numerical correlation heatmap

1. **What we are checking:** Pairwise linear correlations among the selected numerical, non-demographic variables and the binary target.

2. **Why it matters in a credit-risk problem:** Correlations can flag fields that move together and help identify repeated monthly measures, which matters for feature design and interpretation.

3. **The code:** The code uses Pearson correlation on `X1`, `X6`–`X23`, and `Y`; it intentionally excludes `ID` and demographic columns `X2`–`X5`.

4. **How to read the output:** Values near +1 or -1 show stronger linear association; values near 0 show weak linear association. Look for blocks of similar monthly variables.

5. **One honest business insight or limitation:** Correlation is not causation, does not capture all non-linear patterns, and should not be treated as a decision rule.

In [ ]:
numeric_eda_columns = ['X1'] + status_columns + bill_columns + payment_columns + ['Y']
correlations = df[numeric_eda_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(correlations, cmap='vlag', center=0, vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={'label': 'Pearson correlation'})
ax.set_title('Correlation heatmap: selected numerical credit-risk variables')
save_figure('08_numerical_correlation_heatmap.png')

correlations['Y'].drop('Y').sort_values(ascending=False).to_frame('correlation_with_Y')

## EDA 9 — Default versus non-default group comparison

1. **What we are checking:** Means and medians of granted credit, repayment status, bill amounts, and payment amounts for the two recorded outcome groups.

2. **Why it matters in a credit-risk problem:** Group comparison gives a compact starting point for understanding which pre-outcome measures differ in this historical sample.

3. **The code:** The code groups by `Y`, computes means and medians, and plots selected latest-month measures. It deliberately omits demographic fields.

4. **How to read the output:** Compare both mean and median: a large gap often signals skewed monetary data. The chart is a summary, not an individual-level explanation.

5. **One honest business insight or limitation:** Differences between groups can guide later feature exploration, but do not prove that any one variable caused the outcome.

In [ ]:
comparison_columns = ['X1', 'X6'] + bill_columns + payment_columns
group_comparison = df.groupby('Y')[comparison_columns].agg(['mean', 'median']).T
group_comparison.index = [f'{feature} — {statistic}' for feature, statistic in group_comparison.index]
group_comparison = group_comparison.rename(columns={0: 'No default', 1: 'Default next month'})
display(group_comparison.style.format('{:,.1f}'))

selected_comparison = pd.DataFrame({
    'No default': [df.loc[df.Y == 0, 'X1'].median(), df.loc[df.Y == 0, 'X6'].mean(), df.loc[df.Y == 0, 'X12'].median(), df.loc[df.Y == 0, 'X18'].median()],
    'Default next month': [df.loc[df.Y == 1, 'X1'].median(), df.loc[df.Y == 1, 'X6'].mean(), df.loc[df.Y == 1, 'X12'].median(), df.loc[df.Y == 1, 'X18'].median()],
}, index=['Median credit limit (NT$)', 'Mean latest status code', 'Median latest bill (NT$)', 'Median latest payment (NT$)'])
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (label, values) in zip(axes.flat, selected_comparison.iterrows()):
    ax.bar(values.index, values.values, color=[COLORS['No default'], COLORS['Default next month']])
    ax.set_title(label)
    ax.tick_params(axis='x', rotation=12)
save_figure('09_default_group_comparison.png')

## EDA Findings

### Data-backed findings

1. The recorded default-payment outcome is imbalanced: **6,636 of 30,000 records (22.12%)** have `Y=1`; an always-no-default baseline would miss every observed default.
2. Granted credit is right-skewed: the overall median is **NT$140,000**, while the mean is **NT$167,484**.
3. In this historical sample, the median granted credit is **NT$90,000** for records with `Y=1` and **NT$150,000** for records with `Y=0`. This is an association, not evidence that changing a limit changes default risk.
4. Observed default rates decline across the chosen credit-limit bands: **36.07%** below NT$50k, **26.01%** at NT$50k–99,999, **20.77%** at NT$100k–199,999, and **15.13%** at NT$200k or more.
5. The latest documented repayment-delay codes have notably higher observed outcome rates: for `X6`, code `1` has **33.95%** and code `2` has **69.14%**, compared with **12.81%** for the observed but undocumented code `0`. The latter comparison must not be given a business meaning without documentation.
6. Monetary fields are highly skewed, so median and distribution views are more informative than means alone. Median recorded payments are lower for `Y=1` across the six months; for example, the September median is **NT$1,636** versus **NT$2,460** for `Y=0`.
7. Monthly repayment-status fields and monthly monetary fields form related time-series blocks, so later feature engineering should consider redundancy and avoid blindly adding many near-duplicate variables.

### Limitations and cautions

1. These are historical Taiwan credit-card records from 2005, not a current bank portfolio or an Indian lending dataset.
2. UCI’s supplied documentation does not define repayment-status codes `0` and `-2`; this notebook preserves them without inventing labels.
3. Correlation and group differences are descriptive associations, not causation. The dataset does not provide enough timing detail to treat same-month bill and payment amounts as a validated repayment ratio.

## Feature-engineering ideas justified by this EDA

Use these only in a later, train-only feature-engineering module with validation checks:

1. **Repayment-history summaries:** count documented positive delay codes and calculate the maximum documented delay across `X6`–`X11`; preserve `0` and `-2` as separate unknown categories rather than converting them to a guessed delay.
2. **Repayment-status recency features:** retain the latest status (`X6`) and compare it with earlier observed codes to represent documented changes over time.
3. **Six-month bill and payment summaries:** median, mean, minimum, maximum, and trend measures across each sequence, calculated without matching a bill to a payment within the same month.
4. **Skew-robust monetary transformations:** signed/log-like transformations or winsorisation assessed only on the training set, because bill and payment amounts are highly skewed.
5. **Credit-limit bands or non-linear treatment:** test a carefully documented binned or non-linear representation of `X1`, since its relationship with the outcome is not obviously linear.

### Tempting features to avoid for now

- **`X18 / X12` or similar same-month payment-to-bill ratios:** UCI does not document the within-month ordering or pairing.
- **‘Current balance’ = bill minus payment:** the data is not a transaction ledger and does not establish a current balance.
- **Treating `0` or `-2` as on-time payment statuses:** their meanings are undocumented in the supplied metadata.
- **Demographic-variable features (`X2`–`X5`):** outside this EDA module and require a dedicated fairness and governance review.
- **`ID`:** it is an identifier, not customer credit behaviour.

## Feature-engineering preview

This is a preview of the reusable, leakage-aware feature function in `src/features.py`; the feature calculations are intentionally not copied into the notebook.

1. **What we are checking:** The names and first rows of the new per-record features, alongside the frozen baseline feature count.

2. **Why it matters in a credit-risk problem:** A feature must represent information available before the recorded future outcome and be understandable enough to review.

3. **The code:** Import `build_engineered_features()` and `ENGINEERED_FEATURES`, then display a small preview. The function does not read `Y`.

4. **How to read the output:** Each row remains one historical account record. Features beginning `repayment_`, `bill_`, and `payment_` are six-month summaries; `X1` and raw history remain available.

5. **One honest business insight or limitation:** These are candidate inputs, not proven improvements. The later comparison must fit any learned transformation on training data only and evaluate the same untouched validation/test splits fairly.


In [ ]:
from features import ENGINEERED_FEATURES, ENGINEERED_MODEL_FEATURES, MODEL_FEATURES, build_engineered_features

engineered_preview = build_engineered_features(df)
print(f"Frozen baseline features: {len(MODEL_FEATURES)}")
print(f"New engineered summaries: {len(ENGINEERED_FEATURES)}")
print(f"Candidate fair-comparison features: {len(ENGINEERED_MODEL_FEATURES)}")
display(engineered_preview[ENGINEERED_FEATURES].head())
